# Conversational Clustering — Week 1: End-to-end skeleton

**Goal of this notebook:** prove the boring pipeline works on real data. No LLM, no conversation, no operations yet. Just:

1. Pull ~200 astro-ph abstracts from arXiv
2. Embed them with a sentence-transformer
3. Run k-means
4. Eyeball the clusters

If at the end you can look at each cluster and roughly say what it's "about," the foundation is real and you can move on to Week 2 (LLM-as-clusterer baseline).

**Project context:** Solo KDD course project. Hypothesis (to be sharpened later): *Conversational refinement using a structured operation vocabulary converges to a hidden target clustering in fewer turns and with higher final ARI than iterated one-shot LLM clustering, on astro-ph abstracts when the target clustering is non-topical.*


## 0. Setup

Install dependencies if needed. Uncomment the cell below the first time you run.


In [ ]:
# !pip install arxiv sentence-transformers scikit-learn pandas numpy

In [ ]:
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import arxiv
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Reproducibility — set this at the top of every notebook
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
ABSTRACTS_PATH = DATA_DIR / "astro_ph_abstracts.json"


## 1. Pull astro-ph abstracts from arXiv

We pull ~200 recent abstracts across all astro-ph subcategories. We save to disk so we don't re-pull every time we restart the kernel — arXiv rate-limits and it's slow.

**Note on the subcategory tag:** the `arxiv` library returns the *primary* category as `result.primary_category` (e.g. `astro-ph.GA`). We'll use this later as the "easy" hidden target clustering for the simulated user.


In [ ]:
def fetch_astro_ph_abstracts(n=200, force_refresh=False):
    """Pull n recent astro-ph abstracts from arXiv. Cached to disk."""
    if ABSTRACTS_PATH.exists() and not force_refresh:
        print(f"Loading cached abstracts from {ABSTRACTS_PATH}")
        with open(ABSTRACTS_PATH) as f:
            return json.load(f)

    print(f"Fetching {n} abstracts from arXiv...")
    search = arxiv.Search(
        query="cat:astro-ph.*",
        max_results=n,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=3)

    records = []
    for i, result in enumerate(client.results(search)):
        records.append({
            "id": result.entry_id.split("/")[-1],
            "title": result.title.strip().replace("\n", " "),
            "abstract": result.summary.strip().replace("\n", " "),
            "primary_category": result.primary_category,
            "categories": result.categories,
            "published": result.published.isoformat(),
            "authors": [a.name for a in result.authors][:5],  # cap
        })
        if (i + 1) % 50 == 0:
            print(f"  fetched {i+1}/{n}")

    with open(ABSTRACTS_PATH, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Saved {len(records)} abstracts to {ABSTRACTS_PATH}")
    return records

abstracts = fetch_astro_ph_abstracts(n=200)
print(f"\nTotal abstracts: {len(abstracts)}")


In [ ]:
# Quick sanity check — what do we have?
df = pd.DataFrame(abstracts)
print(f"Shape: {df.shape}")
print(f"\nPrimary category distribution:")
print(df["primary_category"].value_counts())
print(f"\nAbstract length stats (chars):")
print(df["abstract"].str.len().describe().round(0))


In [ ]:
# Eyeball the first few — do these actually look like astro-ph?
for rec in abstracts[:3]:
    print(f"[{rec['primary_category']}] {rec['title']}")
    print(f"  {rec['abstract'][:200]}...")
    print()


## 2. Embed the abstracts

Using `all-MiniLM-L6-v2` — small (80MB), fast on CPU, 384-dim embeddings. Good enough for Week 1. We can switch to a stronger model (e.g. `all-mpnet-base-v2` or an OpenAI embedding) later if the clustering is bad.

We embed `title + abstract` together. The title carries a lot of signal in academic papers.


In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")


In [ ]:
# Embed title + abstract concatenated
texts = [f"{rec['title']}. {rec['abstract']}" for rec in abstracts]

t0 = time.time()
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
print(f"\nEmbedded {len(texts)} abstracts in {time.time()-t0:.1f}s")
print(f"Embeddings shape: {embeddings.shape}")


## 3. Cluster with k-means

Starting K=5. The arXiv astro-ph taxonomy has 6 main subcategories (GA, SR, CO, EP, HE, IM), but our 200-paper sample probably has skewed coverage and IM is small, so K=5 is a reasonable starting point.

**This K is a placeholder.** In the real experiment, the LLM-baseline picks K and the conversational system can change K via the `change_k` operation.


In [ ]:
K = 5
kmeans = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

# Internal validity — sanity check, not the real evaluation
sil = silhouette_score(embeddings, cluster_ids, metric="cosine")
print(f"K={K}, silhouette (cosine) = {sil:.3f}")
print(f"Cluster sizes: {Counter(cluster_ids)}")


## 4. Eyeball the clusters

For each cluster, print:
- size
- distribution of arXiv primary categories within it (this is the "easy hidden target" — does k-means roughly recover it?)
- 5 random titles

This is the **whole point of the Week 1 exercise.** Look at this output and ask: do these clusters look like coherent topics? If yes, the foundation is real. If no, something is wrong (bad embeddings, bad K, weird sample) and you fix it before moving on.


In [ ]:
def show_clusters(cluster_ids, abstracts, n_titles=5):
    df = pd.DataFrame(abstracts)
    df["cluster"] = cluster_ids

    for c in sorted(df["cluster"].unique()):
        sub = df[df["cluster"] == c]
        print(f"=== Cluster {c} (n={len(sub)}) ===")
        print(f"  Primary category mix: {dict(sub['primary_category'].value_counts())}")
        sample = sub.sample(min(n_titles, len(sub)), random_state=RANDOM_SEED)
        for _, row in sample.iterrows():
            print(f"  [{row['primary_category']}] {row['title'][:100]}")
        print()

show_clusters(cluster_ids, abstracts)


## 5. Quick numerical check: do clusters align with arXiv categories?

This is *not* the real evaluation — it's a sanity check. If our clusters wildly disagree with arXiv categories, either the embeddings are bad or our sample is too small/skewed. If they agree perfectly, the task is too easy and there's nothing for conversational refinement to study (this is the worry from the project plan — we addressed this by planning to use *non-topical* hidden targets later).

We use ARI (Adjusted Rand Index): 0 = random, 1 = perfect agreement, can be negative.


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

primary_cats = [rec["primary_category"] for rec in abstracts]
ari = adjusted_rand_score(primary_cats, cluster_ids)
nmi = normalized_mutual_info_score(primary_cats, cluster_ids)
print(f"ARI vs arXiv primary category: {ari:.3f}")
print(f"NMI vs arXiv primary category: {nmi:.3f}")
print()
print("Interpretation:")
print("  ARI ~0.0  : clustering ignores arXiv categories")
print("  ARI ~0.3  : weak alignment — interesting regime for the experiment")
print("  ARI ~0.6+ : strong alignment — task may be too easy for topic-target")
print("              (good — this is why we'll use NON-topical hidden targets)")


## 6. Optional: sweep K

Quick sweep to see how silhouette and ARI move with K. Not part of the real experiment, just orientation.


In [ ]:
results = []
for k in [3, 4, 5, 6, 7, 8, 10]:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    cids = km.fit_predict(embeddings)
    results.append({
        "k": k,
        "silhouette": silhouette_score(embeddings, cids, metric="cosine"),
        "ari_vs_arxiv": adjusted_rand_score(primary_cats, cids),
        "nmi_vs_arxiv": normalized_mutual_info_score(primary_cats, cids),
    })
pd.DataFrame(results).round(3)


## Week 1 checklist

- [ ] arXiv pull works and is cached to disk
- [ ] Embedding model loads and runs
- [ ] k-means produces non-degenerate clusters (no cluster has ~all the points, no cluster is empty)
- [ ] Eyeballing the clusters: each cluster has a recognizable theme to a human reader
- [ ] ARI vs arXiv categories is *somewhere in the middle* (not 0, not 1) — confirms the task has signal but isn't trivial

If all five boxes are checked, you're done with Week 1. **Stop here.** Do not start building the LLM baseline tonight — sit with what you have, look at the clusters, and let questions surface naturally before Week 2.

## Week 2 preview (do not start yet)

- Wire up an LLM API (Anthropic / OpenAI)
- Build `oneshot_cluster(abstracts, k) -> dict[abstract_id, cluster_id]` — the baseline
- Decide: does the LLM see all 200 abstracts at once (titles only? title + first sentence?) or does it work over our embeddings (e.g. cluster labels only)?

## Things to write down NOW while it's fresh

A `notes.md` next to this notebook with:
- Anything weird in the data (duplicates? non-English abstracts? cross-listed papers in multiple categories?)
- Clusters that surprised you — good or bad
- Questions you can't answer yet — these become the study-plan refinements
